In [ ]:
# Workflow

# 1. Local: run `youtube_audio_downloader` → `./data` (raw audio)
# 2. For large datasets (e.g., 100+ hours), distribute full length audio files across `part1`, `part2`, ... under `./data` to reduce Colab timeout/disconnect risks
# 3. Upload `./data` to Google Drive (`PROJECT_DIR/data`)
# 4. Colab: run `batch_process.ipynb` segment audio files from each part → `PROJECT_DIR/segments/partN`
# 5. Colab: run `merge_segment_parts.ipynb` to merge all segment folders → `PROJECT_DIR/segments/all_files`
# 6. Colab: run `normalization_and_hf_push.ipynb` to text normalization → HF dataset creation and upload

In [ ]:
#CELL1 - Install Dependencies
!pip install -q torch torchaudio torchcodec
!pip install -q faster-whisper
!pip install -q librosa soundfile
!pip install -q silero-vad
!pip install -q tqdm

In [ ]:
#CELL2 - Imports and Configuration
import os
import torch
import torchaudio
import torchcodec
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
import gc
from typing import List, Tuple, Optional, Generator
import warnings
import time
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

# Configuration
# Project root — folder under MyDrive (upload local ./data here)
PROJECT_DIR = "/content/drive/MyDrive/project_name"  # edit path

# Raw audio input (youtube_audio_downloader output, uploaded to Drive)
INPUT_DIR = f"{PROJECT_DIR}/data/partn"  # edit path

# Segmentation output — use part1, part2, ... for each batch run
OUTPUT_DIR = f"{PROJECT_DIR}/segments/partn"  # edit path

# Segmentation parameters
TARGET_SAMPLE_RATE = 24000  # Required for SNAC
MIN_SEGMENT_DURATION = 3.0  # seconds — minimum duration in final dataset
MAX_SEGMENT_DURATION = 15.0  # seconds

# VAD parameters - Optimized for clean audiobook TTS training
VAD_THRESHOLD = 0.4  # More selective (ideal for clean audiobook audio)
MIN_SILENCE_DURATION_MS = 350
SPEECH_PAD_MS = 0  # Minimal padding (avoid extra audio at segment start)

# Whisper parameters
WHISPER_MODEL = "large-v3"
WHISPER_COMPUTE_TYPE = "float16"
WHISPER_LANGUAGE = "en"  # ISO 639-1 code (e.g. "en", "de", "fr") — change for your audio language

# Processing parameters for large files
CHUNK_DURATION_SEC = 600  # Process 10 minutes at a time to save memory
SAVE_EVERY_N_SEGMENTS = 100  # Save progress periodically

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Target sample rate: {TARGET_SAMPLE_RATE} Hz")
print(f"Segment duration: {MIN_SEGMENT_DURATION}-{MAX_SEGMENT_DURATION} seconds")
print(f"Chunk size for processing: {CHUNK_DURATION_SEC/60:.0f} minutes")

In [ ]:
#CELL3 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#CELL4 - Initialize Models
vad_model = None
whisper_model = None
get_speech_timestamps = None
read_audio_vad = None

def init_models():
    """Initialize models (call once)."""
    global vad_model, whisper_model, get_speech_timestamps, read_audio_vad

    if vad_model is None:
        print("Loading VAD model...")
        model, utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False,
            onnx=False
        )
        (get_speech_timestamps, _, read_audio_vad, _, _) = utils
        vad_model = model.cuda()
        print("VAD model loaded.")

    if whisper_model is None:
        print("Loading Whisper model...")
        from faster_whisper import WhisperModel
        whisper_model = WhisperModel(
            WHISPER_MODEL,
            device="cuda",
            compute_type=WHISPER_COMPUTE_TYPE
        )
        print("Whisper model loaded.")

init_models()
print("Models ready.")


In [ ]:
#CELL5 - Utility Functions
def get_audio_info(audio_path: str) -> Tuple[int, int, float]:
    """Get audio info (sample_rate, num_frames, duration) - compatible with all torchaudio versions."""
    try:
        # Try newer torchaudio API first
        info = torchaudio.info(audio_path)
        return info.sample_rate, info.num_frames, info.num_frames / info.sample_rate
    except (AttributeError, RuntimeError):
        # Fallback: Use backend-specific method to get metadata without loading audio
        try:
            # Try using sox_io backend metadata
            import torchaudio.backend.sox_io_backend as sox_backend
            info = sox_backend.info(audio_path)
            return info.sample_rate, info.num_frames, info.num_frames / info.sample_rate
        except:
            try:
                # Try soundfile as another fast option
                import soundfile as sf
                info = sf.info(audio_path)
                return info.samplerate, info.frames, info.duration
            except:
                # Last resort: load only first frame to get sample rate, then calculate
                waveform, sr = torchaudio.load(audio_path, num_frames=1)
                # Get file size to estimate total frames (this is fast)
                import wave
                try:
                    with wave.open(audio_path, 'rb') as wf:
                        num_frames = wf.getnframes()
                        return sr, num_frames, num_frames / sr
                except:
                    # If all else fails, load the whole file (slowest option)
                    waveform, sr = torchaudio.load(audio_path)
                    num_frames = waveform.shape[1]
                    return sr, num_frames, num_frames / sr

def get_audio_duration(audio_path: str) -> float:
    """Get audio duration without loading entire file."""
    _, _, duration = get_audio_info(audio_path)
    return duration

def load_audio_chunk(audio_path: str, start_sec: float, duration_sec: float,
                     target_sr: int = TARGET_SAMPLE_RATE) -> Tuple[torch.Tensor, int]:
    """Load a specific chunk of audio file."""
    orig_sr, total_frames, _ = get_audio_info(audio_path)

    start_frame = int(start_sec * orig_sr)
    num_frames = int(duration_sec * orig_sr)

    # Ensure we don't read past end of file
    num_frames = min(num_frames, total_frames - start_frame)
    if num_frames <= 0:
        return torch.tensor([]), target_sr

    waveform, sr = torchaudio.load(audio_path, frame_offset=start_frame, num_frames=num_frames)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    return waveform.squeeze(0), target_sr

def format_time(seconds: float) -> str:
    """Format seconds as HH:MM:SS."""
    return str(timedelta(seconds=int(seconds)))

print("Utility functions ready.")


In [ ]:
#CELL6 - VAD Processing (Chunked for Large Files)
def get_speech_segments_chunked(audio_path: str) -> List[dict]:
    """
    Get speech segments from a large audio file using chunked processing.
    Processes in chunks to handle multi-hour files.
    """
    total_duration = get_audio_duration(audio_path)
    print(f"  Total duration: {format_time(total_duration)}")

    all_segments = []
    chunk_size = CHUNK_DURATION_SEC
    overlap = 2.0  # 2 second overlap to avoid cutting speech at boundaries

    num_chunks = int(np.ceil(total_duration / (chunk_size - overlap)))

    # Get audio info once
    orig_sr, total_frames, _ = get_audio_info(audio_path)

    for i in tqdm(range(num_chunks), desc="  VAD processing"):
        chunk_start = i * (chunk_size - overlap)
        chunk_end = min(chunk_start + chunk_size, total_duration)

        # Load chunk at 16kHz for VAD
        start_frame = int(chunk_start * orig_sr)
        num_frames = int((chunk_end - chunk_start) * orig_sr)
        num_frames = min(num_frames, total_frames - start_frame)

        if num_frames <= 0:
            continue

        waveform, sr = torchaudio.load(audio_path, frame_offset=start_frame, num_frames=num_frames)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample to 16kHz for VAD
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            waveform = resampler(waveform)

        wav_16k = waveform.squeeze(0).cuda()

        # Get speech timestamps for this chunk
        try:
            speech_timestamps = get_speech_timestamps(
                wav_16k,
                vad_model,
                threshold=VAD_THRESHOLD,
                min_silence_duration_ms=MIN_SILENCE_DURATION_MS,
                speech_pad_ms=SPEECH_PAD_MS,
                sampling_rate=16000
            )
        except Exception as e:
            print(f"    Warning: VAD failed for chunk {i}: {e}")
            continue

        # Convert to absolute timestamps
        for ts in speech_timestamps:
            abs_start = chunk_start + ts['start'] / 16000
            abs_end = chunk_start + ts['end'] / 16000

            # Skip if this segment was already captured in previous chunk (overlap region)
            if all_segments and abs_start < all_segments[-1]['end'] - 0.1:
                continue

            all_segments.append({
                'start': abs_start,
                'end': abs_end,
                'duration': abs_end - abs_start
            })

        # Clear GPU memory
        del wav_16k
        torch.cuda.empty_cache()

    return all_segments

print("VAD processing ready.")


In [ ]:
#CELL7 - Segment Processing
def merge_segments(segments: List[dict], min_dur: float, max_dur: float, target_dur: float = 10.0) -> List[dict]:
    if not segments:
        return []

    merged = []
    current = segments[0].copy()

    for i in range(1, len(segments)):
        next_seg = segments[i]
        gap = next_seg['start'] - current['end']
        potential_duration = next_seg['end'] - current['start']

        should_merge = False
        if potential_duration <= max_dur and current['duration'] < target_dur:  # Stop merging once target duration is reached
            if gap < 1.5 or current['duration'] < min_dur:
                should_merge = True

        if should_merge:
            current['end'] = next_seg['end']
            current['duration'] = current['end'] - current['start']
        else:
            merged.append(current)
            current = next_seg.copy()

    merged.append(current)
    return merged

def split_long_segment(segment: dict, max_dur: float, target_dur: float = 10.0) -> List[dict]:
    """Split long segments."""
    if segment['duration'] <= max_dur:
        return [segment]

    num_parts = int(np.ceil(segment['duration'] / target_dur))
    part_duration = segment['duration'] / num_parts

    parts = []
    for i in range(num_parts):
        part_start = segment['start'] + i * part_duration
        part_end = segment['start'] + (i + 1) * part_duration if i < num_parts - 1 else segment['end']
        parts.append({
            'start': part_start,
            'end': part_end,
            'duration': part_end - part_start
        })
    return parts

def process_segments(raw_segments: List[dict]) -> List[dict]:
    """Process raw VAD segments."""
    merged = merge_segments(raw_segments, MIN_SEGMENT_DURATION, MAX_SEGMENT_DURATION)

    final = []
    for seg in merged:
        if seg['duration'] > MAX_SEGMENT_DURATION:
            final.extend(split_long_segment(seg, MAX_SEGMENT_DURATION))
        else:
            final.append(seg)

    # Rescue very short segments
    result = []
    i = 0
    while i < len(final):
        seg = final[i].copy()
        if seg['duration'] < MIN_SEGMENT_DURATION and i < len(final) - 1:
            next_seg = final[i + 1]
            gap = next_seg['start'] - seg['end']
            combined_dur = next_seg['end'] - seg['start']
            if gap < 2.0 and combined_dur <= MAX_SEGMENT_DURATION:
                seg['end'] = next_seg['end']
                seg['duration'] = combined_dur
                i += 1
        result.append(seg)
        i += 1

    return result

print("Segment processing ready.")


In [ ]:
#CELL8 - Audio Processing
def trim_silence_gentle(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """Trim silence from edges - optimized for TTS alignment."""
    frame_len = int(0.01 * sr)  # 10ms frames
    threshold = 10 ** (-40 / 20)  # -40dB (more aggressive, suitable for clean audio)

    start_idx = 0
    for i in range(0, len(waveform) - frame_len, frame_len):
        if torch.abs(waveform[i:i + frame_len]).max().item() > threshold:
            start_idx = i  # Reduced padding (was: i - frame_len)
            break

    end_idx = len(waveform)
    for i in range(len(waveform) - frame_len, 0, -frame_len):
        if torch.abs(waveform[i:i + frame_len]).max().item() > threshold:
            end_idx = min(len(waveform), i + frame_len)  # Reduced padding (was: i + 2*frame_len)
            break

    trimmed = waveform[start_idx:end_idx]
    if len(trimmed) < int(MIN_SEGMENT_DURATION * sr):
        return waveform  # Return original if trim shortened segment too much
    return trimmed


def normalize_audio(waveform: torch.Tensor, target_db: float = -3.0) -> torch.Tensor:
    """Normalize audio."""
    rms = torch.sqrt(torch.mean(waveform ** 2))
    if rms > 0:
        target_rms = 10 ** (target_db / 20)
        waveform = waveform * (target_rms / rms)
        max_val = torch.abs(waveform).max()
        if max_val > 0.99:
            waveform = waveform * (0.99 / max_val)
    return waveform

print("Audio processing ready.")


In [ ]:
#CELL9 - Transcription
def transcribe_segment(audio_array: np.ndarray, sr: int) -> str:
    """Transcribe audio."""
    if isinstance(audio_array, torch.Tensor):
        audio_array = audio_array.numpy()

    audio_array = audio_array.astype(np.float32)

    if sr != 16000:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)

    segments, _ = whisper_model.transcribe(
        audio_array,
        language=WHISPER_LANGUAGE,
        task="transcribe",
        beam_size=5,
        best_of=5,
        patience=1.0,
        length_penalty=1.0,
        temperature=0.0,
        compression_ratio_threshold=2.4,
        log_prob_threshold=-1.0,
        no_speech_threshold=0.6,
        condition_on_previous_text=False,
        word_timestamps=False,
        vad_filter=False
    )

    return " ".join([seg.text.strip() for seg in segments]).strip()

print("Transcription ready.")


In [ ]:
#CELL10 - Validation
def validate_segment(waveform: torch.Tensor, sr: int, transcript: str) -> Tuple[bool, str]:
    """Minimal validation."""
    duration = len(waveform) / sr

    # No segments below MIN_SEGMENT_DURATION in final dataset
    if duration < MIN_SEGMENT_DURATION:
        return False, "too_short"

    if not transcript or len(transcript.strip()) == 0:
        return False, "empty_transcript"

    rms = torch.sqrt(torch.mean(waveform ** 2)).item()
    if rms < 0.0001:
        return False, "silent"

    return True, "valid"

print("Validation ready.")



In [ ]:
#CELL11 - Progress Tracking and Resume Support (UPDATED)
def get_next_file_index(output_dir: str) -> int:
    """Get the next available file index based on ACTUAL files on disk."""
    if not os.path.exists(output_dir):
        return 1
    existing = [f for f in os.listdir(output_dir) if f.endswith('.wav')]
    if not existing:
        return 1
    # Extract numeric IDs from filenames
    indices = []
    for f in existing:
        try:
            idx = int(f.split('.')[0])
            indices.append(idx)
        except ValueError:
            continue

    return max(indices) + 1 if indices else 1

def load_progress(output_dir: str) -> dict:
    """Load processing progress."""
    progress_file = os.path.join(output_dir, "_batch_progress.json")
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            return json.load(f)
    return {
        'completed_files': [],
        'failed_files': [],
        'total_segments': 0,
        'total_output_duration': 0
    }

print("Progress tracking ready (Robust Mode).")



In [ ]:
#CELL12 - Main Processing Pipeline (UPDATED - SMART RESUME)
def process_audio_file(input_path: str, output_dir: str, global_idx_start: int) -> Tuple[dict, int]:
    """
    Process a single audio file with checking existing files to skip reprocessing.
    """
    filename = Path(input_path).stem
    print(f"\n{'='*60}")
    print(f"Processing: {filename}")
    print(f"Start Index: {global_idx_start}")
    print(f"{'='*60}")

    os.makedirs(output_dir, exist_ok=True)

    # Get total duration
    total_duration = get_audio_duration(input_path)
    start_time = time.time()

    # Step 1: VAD (fast enough to run every time)
    # VAD is deterministic — same file yields the same segments each run.
    print("\n[1/4] Running VAD...")
    raw_segments = get_speech_segments_chunked(input_path)

    # Step 2: Process segments
    print("\n[2/4] Processing segments...")
    final_segments = process_segments(raw_segments)
    print(f"  Final segments to process: {len(final_segments)}")

    # Step 3: Extract, transcribe, and save (SMART SKIP ADDED)
    print("\n[3/4] Extracting and transcribing...")

    results = []
    skipped_stats = {'too_short': 0, 'empty_transcript': 0, 'silent': 0, 'already_exists': 0}

    # current_idx starts at global_idx_start for this file and increments.
    # IMPORTANT: Indexing stays consistent assuming files are processed in order.
    current_idx = global_idx_start

    batch_size = 50
    num_batches = int(np.ceil(len(final_segments) / batch_size))

    for batch_idx in range(num_batches):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, len(final_segments))
        batch_segments = final_segments[batch_start:batch_end]

        pbar = tqdm(batch_segments, desc=f"  Batch {batch_idx+1}/{num_batches}")

        for seg in pbar:
            # Target file paths
            wav_path = os.path.join(output_dir, f"{current_idx}.wav")
            txt_path = os.path.join(output_dir, f"{current_idx}.txt")

            # --- SMART RESUME CHECK ---
            # If file already exists and is valid, skip processing
            if os.path.exists(wav_path) and os.path.exists(txt_path):
                try:
                    # Simple check: treat non-empty files as existing
                    if os.path.getsize(wav_path) > 1000 and os.path.getsize(txt_path) > 0:
                        with open(txt_path, 'r', encoding='utf-8') as f:
                            existing_transcript = f.read().strip()

                        # Add to metadata but skip processing
                        results.append({
                            'id': current_idx,
                            'duration': seg['duration'], # Approximate duration
                            'transcript': existing_transcript,
                            'source_file': filename,
                            'original_start': seg['start'],
                            'original_end': seg['end']
                        })
                        skipped_stats['already_exists'] += 1
                        current_idx += 1
                        continue # Skip to next segment without running Whisper
                except Exception:
                    # Re-process if file is corrupted
                    pass
            # --------------------------

            # Load just this segment
            segment_audio, sr = load_audio_chunk(
                input_path,
                seg['start'],
                seg['duration'] + 0.1,
                TARGET_SAMPLE_RATE
            )

            if len(segment_audio) == 0:
                skipped_stats['too_short'] += 1
                continue

            # Trim and normalize
            segment_audio = trim_silence_gentle(segment_audio, sr)
            segment_audio = normalize_audio(segment_audio)
            segment_duration = len(segment_audio) / sr

            # Transcribe
            transcript = transcribe_segment(segment_audio.numpy(), sr)

            # Validate
            is_valid, reason = validate_segment(segment_audio, sr, transcript)

            if is_valid:
                # Save immediately
                torchaudio.save(wav_path, segment_audio.unsqueeze(0), sr,
                               encoding="PCM_S", bits_per_sample=16)

                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(transcript)

                results.append({
                    'id': current_idx,
                    'duration': segment_duration,
                    'transcript': transcript,
                    'source_file': filename,
                    'original_start': seg['start'],
                    'original_end': seg['end']
                })

                current_idx += 1
            else:
                skipped_stats[reason] = skipped_stats.get(reason, 0) + 1

            # Update progress bar
            pbar.set_postfix({
                'saved': len(results),
                'exist': skipped_stats['already_exists']
            })

        # Clear memory after each batch
        gc.collect()
        torch.cuda.empty_cache()

    # Step 4: Save metadata
    # ... (rest unchanged) ...

    total_valid_duration = sum(r['duration'] for r in results)
    elapsed_time = time.time() - start_time

    stats = {
        'source_file': input_path,
        'filename': filename,
        'processing_time_sec': elapsed_time,
        'skipped': skipped_stats,
        'num_segments': len(results),
        'total_output_duration_sec': total_valid_duration
    }

    # Update metadata after each file completes
    meta_path = os.path.join(output_dir, f"_meta_{filename}.json")
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, current_idx

print("Main pipeline ready (With Resume).")


In [ ]:
#CELL13 - Batch Processing with Resume Support
def find_audio_files(input_dir: str) -> List[str]:
    """Find all audio files in directory."""
    extensions = ['.wav', '.mp3', '.m4a', '.flac', '.ogg', '.opus']
    files = []
    for ext in extensions:
        files.extend(list(Path(input_dir).glob(f'*{ext}')))
        files.extend(list(Path(input_dir).glob(f'*{ext.upper()}')))
    return sorted([str(f) for f in files])

def process_all_files(input_dir: str, output_dir: str, resume: bool = True) -> dict:
    """
    Process all audio files in directory with resume support.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Find all audio files
    audio_files = find_audio_files(input_dir)
    print(f"Found {len(audio_files)} audio files")

    # Calculate total duration
    total_input_duration = 0
    file_durations = {}
    print("\nScanning files...")
    for f in tqdm(audio_files, desc="Scanning"):
        try:
            dur = get_audio_duration(f)
            file_durations[f] = dur
            total_input_duration += dur
        except Exception as e:
            print(f"  Warning: Could not read {f}: {e}")

    print(f"\nTotal input duration: {format_time(total_input_duration)} ({total_input_duration/3600:.1f} hours)")

# Load progress
    progress_file = os.path.join(output_dir, "_batch_progress.json")

    # CRITICAL: Always compute index from disk, do not trust JSON alone
    current_disk_idx = get_next_file_index(output_dir)

    if resume and os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            progress = json.load(f)

        # If JSON index differs from disk index, use disk (source of truth)
        if progress.get('next_global_idx', 1) != current_disk_idx:
            print(f"Warning: Index mismatch fixed. JSON: {progress.get('next_global_idx')}, Disk: {current_disk_idx}")
            progress['next_global_idx'] = current_disk_idx

        print(f"\nResuming from previous run...")
    else:
        progress = {
            'completed_files': [],
            'failed_files': [],
            'next_global_idx': current_disk_idx, # Start from disk
            'total_segments': 0,
            'total_output_duration': 0
        }
    # Process each file
    all_stats = []
    start_time = time.time()

    for i, audio_path in enumerate(audio_files):
        filename = Path(audio_path).stem

        # Skip if already processed
        if audio_path in progress['completed_files']:
            print(f"\n[{i+1}/{len(audio_files)}] Skipping {filename} (already processed)")
            continue

        print(f"\n\n{'#'*60}")
        print(f"FILE {i+1}/{len(audio_files)}: {filename}")
        print(f"{'#'*60}")

        try:
            stats, next_idx = process_audio_file(
                audio_path,
                output_dir,
                progress['next_global_idx']
            )

            # Update progress
            progress['completed_files'].append(audio_path)
            progress['next_global_idx'] = next_idx
            progress['total_segments'] += stats['num_segments']
            progress['total_output_duration'] += stats['total_output_duration_sec']

            all_stats.append(stats)

        except Exception as e:
            print(f"\nERROR processing {filename}: {e}")
            import traceback
            traceback.print_exc()
            progress['failed_files'].append({'file': audio_path, 'error': str(e)})

        # Save progress after each file
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)

        # Estimate remaining time
        elapsed = time.time() - start_time
        processed_duration = sum(file_durations.get(f, 0) for f in progress['completed_files'])
        remaining_duration = total_input_duration - processed_duration

        if processed_duration > 0:
            rate = elapsed / processed_duration
            eta = remaining_duration * rate
            print(f"\nProgress: {processed_duration/total_input_duration*100:.1f}%")
            print(f"ETA: {format_time(eta)}")

        # Clear memory
        gc.collect()
        torch.cuda.empty_cache()

    # Final summary
    total_elapsed = time.time() - start_time

    print(f"\n\n{'='*60}")
    print("BATCH PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Files processed: {len(progress['completed_files'])}/{len(audio_files)}")
    print(f"Failed files: {len(progress['failed_files'])}")
    print(f"Total segments: {progress['total_segments']}")
    print(f"Total output duration: {format_time(progress['total_output_duration'])}")
    print(f"Total processing time: {format_time(total_elapsed)}")
    if total_elapsed > 0 and total_input_duration > 0:
        print(f"Average speed: {total_input_duration/total_elapsed:.1f}x realtime")
    else:
        print(f"Average speed: N/A")

    if progress['failed_files']:
        print(f"\nFailed files:")
        for f in progress['failed_files']:
            print(f"  - {f['file']}: {f['error']}")

    # Save final summary
    summary = {
        'total_files': len(audio_files),
        'processed_files': len(progress['completed_files']),
        'failed_files': len(progress['failed_files']),
        'total_input_duration_hours': total_input_duration / 3600,
        'total_output_duration_hours': progress['total_output_duration'] / 3600,
        'total_segments': progress['total_segments'],
        'processing_time_hours': total_elapsed / 3600,
        'avg_speed_multiplier': total_input_duration / total_elapsed if total_elapsed > 0 else 0,
        'retention_percent': (progress['total_output_duration'] / total_input_duration) * 100 if total_input_duration > 0 else 0
    }

    with open(os.path.join(output_dir, "final_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)

    return summary

print("Batch processing ready.")


In [ ]:
#CELL14 - Create Final Dataset Files
def create_final_dataset(output_dir: str):
    """Create final metadata and manifest files."""
    print(f"\nCreating final dataset files in {output_dir}...")

    if not os.path.exists(output_dir):
        print("Output directory does not exist!")
        return None

    wav_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.wav') and f[0].isdigit()])

    if not wav_files:
        print("No wav files found in output directory!")
        return None

    metadata = []
    total_duration = 0

    for wav_file in tqdm(wav_files, desc="Building metadata"):
        idx = wav_file.replace('.wav', '')
        txt_path = os.path.join(output_dir, f"{idx}.txt")
        wav_path = os.path.join(output_dir, wav_file)

        if not os.path.exists(txt_path):
            continue

        _, _, duration = get_audio_info(wav_path)
        total_duration += duration

        with open(txt_path, 'r', encoding='utf-8') as f:
            transcript = f.read().strip()

        metadata.append({
            'id': int(idx),
            'wav': wav_file,
            'txt': f"{idx}.txt",
            'duration': duration,
            'transcript': transcript
        })

    if not metadata:
        print("No valid wav/txt pairs found!")
        return None

    # Sort by ID
    metadata.sort(key=lambda x: x['id'])

    # Save metadata.json
    with open(os.path.join(output_dir, "metadata.json"), 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    # Save manifest.jsonl (for training frameworks)
    with open(os.path.join(output_dir, "manifest.jsonl"), 'w', encoding='utf-8') as f:
        for item in metadata:
            f.write(json.dumps({
                'audio_filepath': os.path.join(output_dir, item['wav']),
                'text': item['transcript'],
                'duration': item['duration']
            }, ensure_ascii=False) + '\n')

    print(f"\n✓ Dataset ready!")
    print(f"  Total files: {len(metadata)}")
    print(f"  Total duration: {format_time(total_duration)} ({total_duration/3600:.1f} hours)")
    print(f"  Average duration: {total_duration/len(metadata):.1f}s")
    print(f"  Files: metadata.json, manifest.jsonl")

    return metadata

print("Dataset creation ready.")


In [ ]:
#CELL15 - Run Processing
# Process all files in the input directory
summary = process_all_files(INPUT_DIR, OUTPUT_DIR, resume=True)


In [ ]:
#CELL16 - Create Final Dataset
create_final_dataset(OUTPUT_DIR)

print("\n" + "="*60)
print("ALL DONE!")
print("="*60)

